# Workstream 8: Response Quality EDA

ranking を固定した prediction JSON 同士で、response length、Distinct-1/2、反復率、ranking signature の一致を確認します。Blind response の評価は公開されている prediction JSON の表層品質に限定し、非公開正解は扱いません。


## Setup


In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

try:
    from IPython.display import Image, Markdown, display
except Exception:
    Image = Markdown = display = None

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 160)


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "EDA").exists():
            return candidate
    raise RuntimeError("Repository root was not found from the current working directory.")

ROOT = find_repo_root(Path.cwd())
EDA_DIR = ROOT / "EDA"
TABLE_DIR = EDA_DIR / "tables"
FIGURE_DIR = EDA_DIR / "figures"
SUMMARY_DIR = EDA_DIR / "summary"
EXPERIMENT_DIR = ROOT / "mcrs" / "experiments"
INFERENCE_DIR = ROOT / "exp" / "inference"

for directory in (TABLE_DIR, FIGURE_DIR, SUMMARY_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"ROOT={ROOT}")
print(f"TABLE_DIR={TABLE_DIR}")


In [ ]:
def read_table(name: str, **kwargs) -> pd.DataFrame:
    path = TABLE_DIR / name
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def read_csv_path(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def show_df(df: pd.DataFrame, n: int = 20) -> None:
    if df.empty:
        print("empty dataframe")
        return
    if display is not None:
        display(df.head(n))
    else:
        print(df.head(n).to_string(index=False))


def show_image(name: str) -> None:
    path = FIGURE_DIR / name
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        return
    if display is not None and Image is not None:
        display(Image(filename=str(path)))
    else:
        print(path)


def save_table(df: pd.DataFrame, name: str) -> Path | None:
    if df.empty:
        print(f"skip empty table: {name}")
        return None
    path = TABLE_DIR / name
    df.to_csv(path, index=False)
    print(f"saved: {path.relative_to(ROOT)} ({len(df):,} rows)")
    return path


def barplot(df: pd.DataFrame, *, x: str, y: str, hue: str | None = None, title: str = "", rotate: int = 0, figsize=(10, 4)) -> None:
    if df.empty:
        print("skip empty plot")
        return
    fig, ax = plt.subplots(figsize=figsize)
    if sns is not None:
        sns.barplot(data=df, x=x, y=y, hue=hue, ax=ax)
    else:
        df.plot(kind="bar", x=x, y=y, ax=ax)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=rotate)
    fig.tight_layout()
    plt.show()


## Discover Prediction Files


In [ ]:
def load_prediction_file(path: Path) -> list[dict]:
    try:
        data = json.loads(path.read_text())
    except Exception as exc:
        print(f"skip {path.relative_to(ROOT)}: {exc}")
        return []
    if isinstance(data, dict) and "predictions" in data:
        data = data["predictions"]
    if not isinstance(data, list):
        print(f"skip non-list prediction: {path.relative_to(ROOT)}")
        return []
    return [row for row in data if isinstance(row, dict)]

prediction_paths = []
if INFERENCE_DIR.exists():
    for path in sorted(INFERENCE_DIR.rglob("*.json")):
        if path.name == "prediction.json" or not path.name.startswith("validate_"):
            prediction_paths.append(path)

prediction_index = pd.DataFrame([
    {"file_id": i, "path": path.relative_to(ROOT).as_posix(), "size_kb": round(path.stat().st_size / 1024, 1)}
    for i, path in enumerate(prediction_paths)
])
show_df(prediction_index, 200)


## Response Lexical Metrics


In [ ]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"[A-Za-z0-9']+", text.lower())


def distinct_n(tokens: list[str], n: int) -> float:
    if len(tokens) < n:
        return 0.0
    grams = list(zip(*(tokens[i:] for i in range(n))))
    return len(set(grams)) / len(grams) if grams else 0.0


def repeated_bigram_rate(tokens: list[str]) -> float:
    if len(tokens) < 2:
        return 0.0
    bigrams = list(zip(tokens, tokens[1:]))
    counts = Counter(bigrams)
    repeated = sum(count - 1 for count in counts.values() if count > 1)
    return repeated / len(bigrams) if bigrams else 0.0

length_rows = []
lexical_rows = []
ranking_rows = []
for file_id, path in enumerate(prediction_paths):
    rows = load_prediction_file(path)
    if not rows:
        continue
    all_tokens = []
    lengths = []
    repeated_rates = []
    ranking_signatures = []
    unique_tracks = set()
    malformed_top20 = 0
    for row in rows:
        response = str(row.get("predicted_response", ""))
        tokens = tokenize(response)
        all_tokens.extend(tokens)
        lengths.append(len(tokens))
        repeated_rates.append(repeated_bigram_rate(tokens))
        track_ids = row.get("predicted_track_ids") or []
        if len(track_ids) != 20:
            malformed_top20 += 1
        unique_tracks.update(str(track_id) for track_id in track_ids)
        ranking_signatures.append("|".join(str(track_id) for track_id in track_ids[:20]))
    rel_path = path.relative_to(ROOT).as_posix()
    length_rows.append({
        "file_id": file_id,
        "path": rel_path,
        "rows": len(rows),
        "response_len_mean": float(np.mean(lengths)) if lengths else 0.0,
        "response_len_median": float(np.median(lengths)) if lengths else 0.0,
        "response_len_p95": float(np.percentile(lengths, 95)) if lengths else 0.0,
        "malformed_top20_rows": malformed_top20,
    })
    lexical_rows.append({
        "file_id": file_id,
        "path": rel_path,
        "distinct_1": distinct_n(all_tokens, 1),
        "distinct_2": distinct_n(all_tokens, 2),
        "avg_repeated_bigram_rate": float(np.mean(repeated_rates)) if repeated_rates else 0.0,
        "unique_recommended_tracks": len(unique_tracks),
    })
    ranking_rows.append({
        "file_id": file_id,
        "path": rel_path,
        "rows": len(rows),
        "unique_ranking_signatures": len(set(ranking_signatures)),
    })

response_length_stats = pd.DataFrame(length_rows)
response_lexical_diversity = pd.DataFrame(lexical_rows)
response_ranking_signatures = pd.DataFrame(ranking_rows)

save_table(response_length_stats, "response_length_stats.csv")
save_table(response_lexical_diversity, "response_lexical_diversity.csv")
save_table(response_ranking_signatures, "response_ranking_signatures.csv")

show_df(response_length_stats, 100)
show_df(response_lexical_diversity.sort_values("distinct_2", ascending=False) if not response_lexical_diversity.empty else response_lexical_diversity, 100)


## Fixed-Ranking Response Variant Detection


In [ ]:
variant_rows = []
loaded = []
for file_id, path in enumerate(prediction_paths):
    rows = load_prediction_file(path)
    if not rows:
        continue
    by_key = {}
    for row in rows:
        key = (str(row.get("session_id")), str(row.get("turn_number")))
        ranking = tuple(str(track_id) for track_id in (row.get("predicted_track_ids") or [])[:20])
        by_key[key] = ranking
    loaded.append((file_id, path.relative_to(ROOT).as_posix(), by_key))

for i in range(len(loaded)):
    left_id, left_path, left = loaded[i]
    for j in range(i + 1, len(loaded)):
        right_id, right_path, right = loaded[j]
        common = set(left) & set(right)
        if not common:
            continue
        same = sum(1 for key in common if left[key] == right[key])
        variant_rows.append({
            "left_file_id": left_id,
            "right_file_id": right_id,
            "common_tasks": len(common),
            "same_top20_tasks": same,
            "same_top20_rate": same / len(common),
            "left_path": left_path,
            "right_path": right_path,
        })
response_fixed_ranking_pairs = pd.DataFrame(variant_rows).sort_values("same_top20_rate", ascending=False) if variant_rows else pd.DataFrame()
save_table(response_fixed_ranking_pairs, "response_fixed_ranking_pairs.csv")
show_df(response_fixed_ranking_pairs, 100)


## Quick Plots


In [ ]:
if not response_length_stats.empty:
    plot_df = response_length_stats.sort_values("response_len_mean", ascending=False).head(30)
    barplot(plot_df, x="file_id", y="response_len_mean", title="Mean response length by prediction file", figsize=(12, 4))

if not response_lexical_diversity.empty:
    plot_df = response_lexical_diversity.sort_values("distinct_2", ascending=False).head(30)
    barplot(plot_df, x="file_id", y="distinct_2", title="Distinct-2 by prediction file", figsize=(12, 4))


## Findings / Decisions / Next Actions

- Findings:
- Decisions:
- Next actions:
